# WIC Eligibility Case Generation

The Special Supplemental Nutrition Program for Women, Infants, and Children (WIC) has
distinct rules from SNAP:

- **Income threshold**: 185% FPL (vs. SNAP's 130%)
- **No asset test** — resources are never counted
- **Categorical eligibility** — SNAP/Medicaid/TANF recipients are automatically income-eligible
- **Participant categories**: pregnant, breastfeeding, postpartum (up to 6 months), infant, child under 5
- **Income period**: WIC guidelines run July 1 – June 30, NOT the federal fiscal year

Source: 7 CFR Part 246; USDA FNS WIC IEG 2025-2026 (Federal Register 2025-03576)

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from govsynth.sources.us.wic import WICSource
from govsynth.profiles.us_household import USHouseholdProfile
from govsynth.generators.wic_eligibility import WICEligibilityGenerator

print('Ready.')

## 1. Inspect WIC FY2026 Income Eligibility Guidelines

In [ ]:
source = WICSource(fiscal_year=2026, state='VA')
t = source.thresholds()

print(f'Program: WIC')
print(f'Period: {t.extra["period_label"]} (effective {t.extra["effective_start"]} – {t.extra["effective_end"]})')
print(f'Income threshold: {t.extra["income_limit_pct_fpl"]}% FPL')
print(f'FPL basis: {t.extra.get("fpl_basis_year", 2025)} HHS poverty guidelines')
print(f'Verification: {t.extra["verification_status"].upper()}')
print(f'Source: {t.source}')
print()
print(f'{"HH Size":<10} {"Monthly Limit (185% FPL)":<26} {"Annual"}')
print('-' * 52)
for size in range(1, 9):
    lim = t.by_household_size(size)
    annual = lim.gross_monthly * 12
    print(f'{size:<10} ${lim.gross_monthly:>10,.0f}/month               ${annual:>10,.0f}/year')
print()
print(f'No asset test — resources are not counted (7 CFR 246.7(d)(1))')
print(f'Categorical eligibility: SNAP, Medicaid, or TANF receipt = auto income-eligible')


## 2. SNAP vs. WIC Income Limits — Side by Side

A common model error is conflating SNAP and WIC income thresholds. WIC is significantly more generous.

In [ ]:
from govsynth.sources.us.snap import SNAPSource

snap = SNAPSource(fiscal_year=2026, state='VA')
snap_t = snap.thresholds()

print(f'{"HH Size":<10} {"SNAP Gross (130%)":<22} {"WIC (185%)":<22} {"Difference"}')
print('-' * 70)
for size in range(1, 7):
    snap_lim = snap_t.by_household_size(size).gross_monthly
    wic_lim = t.by_household_size(size).gross_monthly
    diff = wic_lim - snap_lim
    print(f'{size:<10} ${snap_lim:>10,.0f}/month       ${wic_lim:>10,.0f}/month       +${diff:,.0f}')


## 3. Income Boundary Cases — Pregnant Applicant

In [ ]:
hh_size = 2  # pregnant woman + 1 household member
limits_2 = t.by_household_size(hh_size)

print(f'2-person WIC limit: ${limits_2.gross_monthly:,.2f}/month')
print()

offsets = [('5% below', -0.05), ('1% below', -0.01), ('AT LIMIT', 0.0), ('1% above', 0.01), ('5% above', 0.05)]
print(f'{"Label":<12} {"Gross Income":<18} {"WIC Eligible?"}')
print('-' * 48)
for label, pct in offsets:
    profile = USHouseholdProfile.at_threshold(
        program='wic', threshold='income_limit_185pct_fpl',
        state='VA', household_size=hh_size, fiscal_year=2026,
        offset_pct=pct, seed=42,
    )
    eligible, reason = source.is_eligible(
        household_size=profile.household_size,
        monthly_gross_income=profile.monthly_gross_income,
        participant_category='pregnant',
    )
    print(f'{label:<12} ${profile.monthly_gross_income:>10,.2f}       {"ELIGIBLE" if eligible else "INELIGIBLE"}')


## 4. Categorical Eligibility — SNAP Recipient Auto-Qualifies

In [ ]:
# Income 200% FPL — would normally fail WIC's 185% limit
hh_size = 3
limits_3 = t.by_household_size(hh_size)
gross_over = round(limits_3.gross_monthly * 2.00, 2)

print(f'3-person WIC limit: ${limits_3.gross_monthly:,.2f}/month')
print(f'Test income (200% FPL): ${gross_over:,.2f}/month')
print()

# Without categorical eligibility
elig_no_cat, reason = source.is_eligible(
    household_size=hh_size,
    monthly_gross_income=gross_over,
    participant_category='infant',
    is_categorically_eligible=False,
)
print(f'No categorical eligibility: {"ELIGIBLE" if elig_no_cat else "INELIGIBLE"}')
print(f'  Reason: {reason}')
print()

# With SNAP receipt (is_categorically_eligible=True)
elig_snap, reason_snap = source.is_eligible(
    household_size=hh_size,
    monthly_gross_income=gross_over,
    participant_category='infant',
    is_categorically_eligible=True,
)
print(f'SNAP/Medicaid/TANF recipient (auto income-eligible): {"ELIGIBLE" if elig_snap else "INELIGIBLE"}')
print(f'  Reason: {reason_snap}')


## 5. Generate 20 WIC Test Cases

In [ ]:
from govsynth import Pipeline
from collections import Counter

pipeline = Pipeline.from_preset('wic.national')
cases = pipeline.generate(n=20, seed=42)

print(f'Generated {len(cases)} cases')
print(f'Eligible:   {sum(1 for c in cases if c.expected_outcome == "eligible")}')
print(f'Ineligible: {sum(1 for c in cases if c.expected_outcome == "ineligible")}')
print()

diff = Counter(c.difficulty.value for c in cases)
print('Difficulty:', dict(diff))
print()
print('Variation tags in cases:')
all_tags = Counter(tag for c in cases for tag in c.variation_tags)
for tag, n in all_tags.most_common():
    print(f'  {tag:<35} {n}')

## 6. Inspect a WIC Case

In [ ]:
case = cases[0]
print(f'ID: {case.case_id}')
print(f'Difficulty: {case.difficulty.value}')
print()
print('Scenario:')
print(case.scenario.summary)
print()
print('Expected outcome:', case.expected_outcome.upper())
print()
print('Rationale:')
print(case.rationale_trace.to_plain_text())
